**# RoadRisk-Public-Spending-Brazil**
 In this project, I will build a **machine learning model to predict accident volumes and the associated public spending impact on federal highways in Minas Gerais (MG).** It involves supervised learning to forecast monthly patterns for 2026 and **estimate the financial burden of traffic accidents on the state's public resources.**

 I will use the following **pipeline** based on the **CRISP-DM framework:**

**1. Define the business problem.**

**2. Collect the data and get a general overview of it.**

**3. Split the data into train and test sets.**

**4. Explore the data (exploratory data analysis).**

**5. Feature engineering, data cleaning, and preprocessing.**

**6. Model training, comparison, feature selection, and tuning.**

**7. Final production model testing and evaluation.**

**8. Conclude and interpret the model results.**
 
**9. Deploy.**

 In **this notebook**, I will perform exploratory data analysis (EDA), covering steps 1 to 4 of the pipeline above. The main objective here is to uncover insights regarding temporal (seasonality), geographic (highways/segments), and contextual (accident causes) patterns within Minas Gerais. Thus, even before building a model, it will be possible to support stakeholders—such as the State Government, PRF, and SAMU MG—with data-driven profiles for enforcement prioritization and emergency readiness planning. Furthermore, I will approach these steps in detail below, explaining the rationale behind each decision.

# 1 — BUSINESS UNDERSTANDING
## 1. Business Problem
The state of Minas Gerais holds one of the largest and most complex federal highway networks in Brazil, resulting in a critical volume of accidents that strains public funds and emergency services (SAMU and PRF). Currently, the allocation of speed cameras, patrol vehicles, and rescue teams is mostly reactive or based on simple historical averages, without considering seasonality and the direct financial impact of each incident. This lack of predictability prevents the state government and federal agencies from optimizing safety and infrastructure investments, leading to inefficient spending and the loss of lives that could be prevented through data-driven preventive interventions.

## 1.1 What is the context?
The scenario involves road safety management and public health in a strategic territory for Brazilian logistics. The project aims to integrate police occurrence data with financial and predictive impact models to transform highway management into an evidence-based operation.

Severity Rate per Segment: A metric that weights accidents by the number of victims and severity in specific stretches, allowing the identification of where the risk to life is highest, regardless of the total collision volume.

Estimated Operational Cost (EOC): Projected value of public spending (SAMU, road clearing, forensics) per accident type, serving as a basis to justify investments in electronic surveillance.

Electronic Surveillance ROI: The relationship between the implementation cost of speed cameras (CAPEX/OPEX) and the expected reduction in accident spending plus projected revenue, aiming for the fiscal balance of the measure.

## 1.2 Which are the project objectives?
2026 Seasonality Forecast: Generate a monthly forecast of accident volumes for the year 2026 with an accuracy target between 90% and 95%, allowing for advanced planning of work shifts and SAMU resources.

Cost-Benefit Modeling for Speed Cameras: Quantify implementation costs and the expectation of revenue and public savings for new surveillance points.

Impact Simulation of Measures: Create scenarios that demonstrate the potential reduction in accidents and public spending if specific measures (speed cameras, increased patrolling) are applied to identified hotspots.

## 1.3 Which are the project benefits?
Optimization of Public Resources: Precise allocation of funds to stretches where every real invested in prevention generates the greatest savings in hospital and operational costs.

Increased SAMU Readiness: Reduction in response time to serious incidents through strategic positioning of teams based on monthly forecasts.

Transparent and Justifiable Decisions: Provide stakeholders (MG Government and PRF) with technical grounding for speed camera installation, balancing safety and revenue.

Mortality Reduction: Direct decrease in deaths and serious injuries through electronic surveillance in locations with high accident rates due to speeding.

## 1.4 Conclusion — Output Strategy
The final delivery based on predictive models and scenario simulators is superior to static reports because it allows the public manager to interact with the data (e.g., "What if we install 10 speed cameras on BR-381?"). While descriptive analysis only shows the past, a Machine Learning model with 90%+ accuracy offers a budgetary and operational planning tool for 2026, directly connecting the cost of safety with the benefit of preserving lives and public funds.

## NOTEBOOK VISUAL CONFIGURATION

In [8]:
# ============================================================
# RoadRisk-Public-Spending-Brazil — Visual Configuration
# ============================================================

# Data manipulation and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# Global visualization settings
%matplotlib inline

mpl.style.use('ggplot')

mpl.rcParams['axes.facecolor']  = 'white'     # White background for clarity in reports
mpl.rcParams['axes.linewidth']  = 1.2           # Slightly highlighted border
mpl.rcParams['xtick.color']     = '#2c3e50'    # Dark gray ticks for readability
mpl.rcParams['ytick.color']     = '#2c3e50'
mpl.rcParams['grid.color']      = '#ecf0f1'    # Soft grid to avoid clutter
mpl.rcParams['figure.dpi']      = 150          # High resolution for presentations
mpl.rcParams['axes.grid']       = True         # Always show grid
mpl.rcParams['font.size']       = 10           # Base font size

# Project Color Palette — Focus on Safety and Alert
# Justification: 
# - Deep Blue (#001F3F): Represents government authority and stability (PRF/State).
# - Orange Tones: Represent alerts, traffic signaling, and urgency (Radar/SAMU).
# - Gray (#95a5a6): Used for secondary or historical data.
color_palette = [
    '#001F3F', # Navy Blue (Primary - Institutional)
    '#FF851B', # Vibrant Orange (Highlight - Speed Cameras/Alerts)
    '#FF4136', # Soft Red (Emergency - Serious Accidents)
    '#FFB74D', # Pastel Orange (Secondary data/Averages)
    '#39CCCC', # Teal (Cost comparisons)
    '#95a5a6'  # Neutral Gray (Historical context)
]

# Applying as the default seaborn palette
sns.set_palette(sns.color_palette(color_palette))

# Display for visual validation
print("RoadRisk color palette successfully defined!")
sns.color_palette(color_palette)

RoadRisk color palette successfully defined!


[(0.0, 0.12156862745098039, 0.24705882352941178),
 (1.0, 0.5215686274509804, 0.10588235294117647),
 (1.0, 0.2549019607843137, 0.21176470588235294),
 (1.0, 0.7176470588235294, 0.30196078431372547),
 (0.2235294117647059, 0.8, 0.8),
 (0.5843137254901961, 0.6470588235294118, 0.6509803921568628)]

In [10]:
import pandas as pd
import os

file_path = "../data/processed/occ_mg_master.parquet"
file_path = "../data/processed/pers_mg_master.parquet"

if os.path.exists(file_path):
    df = pd.read_parquet(file_path)


In [11]:
df.head()

,id,pesid,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_principal,...,ilesos,feridos_leves,feridos_graves,mortos,latitude,longitude,regional,delegacia,uop,year_reference
0,47.0,45.0,2017-01-01,domingo,04:50:00,MG,381.0,605.2,OLIVEIRA,Sim,...,0.0,1.0,0.0,0.0,-20.636968,-44.735700,SPRF-MG,DEL04-MG,UOP01-DEL04-MG,2017
1,47.0,44.0,2017-01-01,domingo,04:50:00,MG,381.0,605.2,OLIVEIRA,Sim,...,0.0,1.0,0.0,0.0,-20.636968,-44.735700,SPRF-MG,DEL04-MG,UOP01-DEL04-MG,2017
2,47.0,45.0,2017-01-01,domingo,04:50:00,MG,381.0,605.2,OLIVEIRA,Sim,...,0.0,1.0,0.0,0.0,-20.636968,-44.735700,SPRF-MG,DEL04-MG,UOP01-DEL04-MG,2017
3,47.0,44.0,2017-01-01,domingo,04:50:00,MG,381.0,605.2,OLIVEIRA,Sim,...,0.0,1.0,0.0,0.0,-20.636968,-44.735700,SPRF-MG,DEL04-MG,UOP01-DEL04-MG,2017
4,52.0,1528.0,2017-01-01,domingo,05:00:00,MG,262.0,368.0,JUATUBA,Sim,...,0.0,1.0,0.0,0.0,-19.956619,-44.344404,SPRF-MG,DEL01-MG,UOP03-DEL01-MG,2017


In [5]:
import pandas as pd

# Carregando Ocorrências (Geral)
df_occ = pd.read_parquet("../data/processed/occ_mg_master.parquet")

# Carregando Pessoas (Detalhado)
df_pers = pd.read_parquet("../data/processed/pers_mg_master.parquet")

print(f"Total de Ocorrências em MG: {len(df_occ)}")
print(f"Total de Pessoas envolvidas em MG: {len(df_pers)}")

Total de Ocorrências em MG: 83409
Total de Pessoas envolvidas em MG: 561796
